In [2]:
import pandas as pd

df = pd.read_csv(r"D:\Project_Depi\data\labeled\Twitter_Data.csv")

print(df.shape)
df.head()

(162980, 2)


,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [3]:
print(df.columns)
df.info()

Index(['clean_text', 'category'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB


In [4]:
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].head())


Column: clean_text
0    when modi promised “minimum government maximum...
1    talk all the nonsense and continue all the dra...
2    what did just say vote for modi  welcome bjp t...
3    asking his supporters prefix chowkidar their n...
4    answer who among these the most powerful world...
Name: clean_text, dtype: object

Column: category
0   -1.0
1    0.0
2    1.0
3    1.0
4    1.0
Name: category, dtype: float64


In [5]:
def map_label(x):
    if x == -1:
        return "negative"
    elif x == 0:
        return "neutral"
    elif x == 1:
        return "positive"

df["label"] = df["category"].apply(map_label)

df["label"].value_counts()

label
positive    72250
neutral     55213
negative    35510
Name: count, dtype: int64

In [6]:
def map_label(x):
    if x == -1:
        return "negative"
    elif x == 0:
        return "neutral"
    elif x == 1:
        return "positive"

df["label"] = df["category"].apply(map_label)

df["label"].value_counts()

label
positive    72250
neutral     55213
negative    35510
Name: count, dtype: int64

In [14]:
print(df["label"].isnull().sum())
print(df["label"].unique())
print(df["category"].unique())

7
['negative' 'neutral' 'positive' None]
[-1.  0.  1. nan]


In [15]:
df = df[df["category"].notna()].copy()
df = df[df["label"].notna()].copy()

print(df["label"].isnull().sum())
print(df["category"].isnull().sum())
print(df["label"].value_counts())

0
0
label
positive    72250
neutral     55213
negative    35510
Name: count, dtype: int64


In [17]:
df["clean_text"] = df["clean_text"].fillna("").astype(str)
df = df[df["clean_text"].str.strip() != ""].copy()

print(df["clean_text"].isnull().sum())
print((df["clean_text"].str.strip() == "").sum())
print(df.shape)

0
0
(162968, 3)


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(X_train_vec.shape)
print(X_test_vec.shape)

(130374, 5000)
(32594, 5000)


In [21]:
print(df.head(10))

                                          clean_text  category     label
0  when modi promised “minimum government maximum...      -1.0  negative
1  talk all the nonsense and continue all the dra...       0.0   neutral
2  what did just say vote for modi  welcome bjp t...       1.0  positive
3  asking his supporters prefix chowkidar their n...       1.0  positive
4  answer who among these the most powerful world...       1.0  positive
5           kiya tho refresh maarkefir comment karo        0.0   neutral
6  surat women perform yagna seeks divine grace f...       0.0   neutral
7  this comes from cabinet which has scholars lik...       0.0   neutral
8  with upcoming election india saga going import...       1.0  positive
9                         gandhi was gay does modi         1.0  positive


# LogisticRegression

In [24]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9267043014051666

Classification Report:

              precision    recall  f1-score   support

    negative       0.92      0.83      0.87      7102
     neutral       0.92      0.98      0.95     11042
    positive       0.94      0.94      0.94     14450

    accuracy                           0.93     32594
   macro avg       0.92      0.91      0.92     32594
weighted avg       0.93      0.93      0.93     32594



# Naive Bayes

In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

nb_pred = nb_model.predict(X_test_vec)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, nb_pred))

Naive Bayes Accuracy: 0.7425906608578265

Classification Report:

              precision    recall  f1-score   support

    negative       0.91      0.42      0.58      7102
     neutral       0.88      0.67      0.76     11042
    positive       0.66      0.95      0.78     14450

    accuracy                           0.74     32594
   macro avg       0.82      0.68      0.71     32594
weighted avg       0.79      0.74      0.73     32594



# Linear SVM

In [26]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

svm_model = LinearSVC()
svm_model.fit(X_train_vec, y_train)

svm_pred = svm_model.predict(X_test_vec)

print("Linear SVM Accuracy:", accuracy_score(y_test, svm_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, svm_pred))

Linear SVM Accuracy: 0.94213658955636

Classification Report:

              precision    recall  f1-score   support

    negative       0.92      0.87      0.90      7102
     neutral       0.95      0.98      0.96     11042
    positive       0.95      0.95      0.95     14450

    accuracy                           0.94     32594
   macro avg       0.94      0.93      0.94     32594
weighted avg       0.94      0.94      0.94     32594



In [27]:
tests = [
    "Vodafone internet is extremely slow and customer service is terrible",
    "Vodafone offers are great and the network has improved a lot recently",
    "I have been using Vodafone for a few years now",
    "The internet is good sometimes but the support is very bad",
    "I keep losing connection every night and it's really frustrating",
    "bad service"
]

for t in tests:
    print(t)
    print("LR:", model.predict(vectorizer.transform([t]))[0])
    print("NB:", nb_model.predict(vectorizer.transform([t]))[0])
    print("SVM:", svm_model.predict(vectorizer.transform([t]))[0])
    print("-"*50)

Vodafone internet is extremely slow and customer service is terrible
LR: negative
NB: negative
SVM: negative
--------------------------------------------------
Vodafone offers are great and the network has improved a lot recently
LR: positive
NB: positive
SVM: positive
--------------------------------------------------
I have been using Vodafone for a few years now
LR: negative
NB: positive
SVM: negative
--------------------------------------------------
The internet is good sometimes but the support is very bad
LR: positive
NB: positive
SVM: positive
--------------------------------------------------
I keep losing connection every night and it's really frustrating
LR: positive
NB: positive
SVM: positive
--------------------------------------------------
bad service
LR: negative
NB: negative
SVM: negative
--------------------------------------------------


In [28]:
import joblib

joblib.dump(svm_model, "../models/svm_model.pkl")
joblib.dump(vectorizer, "../models/vectorizer.pkl")

print("Model and vectorizer saved successfully")

Model and vectorizer saved successfully
